# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucy96039/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
!git clone https://github.com/lucy96039/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 134 (delta 46), reused 98 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.84 MiB | 10.26 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/flyrank-ml-internship


In [2]:
import os

REPO_URL = "https://github.com/lucy96039/flyrank-ml-internship.git"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

os.chdir(REPO_DIR)

print("✅ Repository ready")
print("Current directory:", os.getcwd())

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 139 (delta 50), reused 98 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.85 MiB | 8.08 MiB/s, done.
Resolving deltas: 100% (50/50), done.
✅ Repository ready
Current directory: /content/flyrank-ml-internship


In [3]:
import os

csv_files = []

for root, dirs, files in os.walk("data"):
    dirs[:] = [d for d in dirs if d != ".git"]

    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("CSV files found:")

for i, file in enumerate(csv_files, 1):
    print(i, "->", file)

CSV files found:
1 -> data/raw/content_refresh_anonymized.csv


In [4]:
import pandas as pd

if not csv_files:
    raise FileNotFoundError(
        "❌ No CSV dataset found inside the data folder."
    )

DATA_PATH = csv_files[0]

df = pd.read_csv(DATA_PATH)

print("✅ Dataset loaded")
print("Path:", DATA_PATH)
print("Shape:", df.shape)

display(df.head())

✅ Dataset loaded
Path: data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
import os

os.makedirs("work/outputs", exist_ok=True)

print("✅ work/outputs is ready")

✅ work/outputs is ready


In [6]:
print("Columns:")

for i, col in enumerate(df.columns):
    print(i, "->", col)

Columns:
0 -> content_id
1 -> client_id
2 -> search_volume
3 -> competition
4 -> competition_level
5 -> cpc
6 -> content_type
7 -> main_intent
8 -> word_count
9 -> char_count
10 -> provider_used
11 -> model_used
12 -> impressions_90d
13 -> clicks_90d
14 -> pageviews_90d
15 -> sessions_90d
16 -> users_90d
17 -> engaged_sessions_90d
18 -> ai_sessions_90d
19 -> scroll_events_90d
20 -> days_with_impressions
21 -> days_with_sessions
22 -> impressions_last_30d
23 -> clicks_last_30d
24 -> sessions_last_30d
25 -> impressions_prev_30d
26 -> clicks_prev_30d
27 -> sessions_prev_30d
28 -> content_age_days
29 -> age_tier
30 -> age_tier_order
31 -> days_since_last_update
32 -> freshness_tier
33 -> word_count_tier
34 -> char_count_tier
35 -> ctr
36 -> avg_position
37 -> engagement_rate
38 -> scroll_rate
39 -> ai_traffic_pct
40 -> impression_tier
41 -> position_tier
42 -> trend_direction
43 -> trend_pct


In [7]:
import numpy as np

# Make sure the two signals are numeric
df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"],
    errors="coerce"
).fillna(0)

df["search_volume"] = pd.to_numeric(
    df["search_volume"],
    errors="coerce"
).fillna(0)

# Normalize staleness
max_stale = df["days_since_last_update"].max()

if max_stale > 0:
    stale_score = (
        df["days_since_last_update"] / max_stale
    )
else:
    stale_score = 0

# Normalize search volume
max_volume = df["search_volume"].max()

if max_volume > 0:
    volume_score = (
        df["search_volume"] / max_volume
    )
else:
    volume_score = 0

# Baseline score
df["baseline_score"] = (
    0.7 * stale_score +
    0.3 * volume_score
)

# Reason code
df["reason_code"] = np.where(
    df["days_since_last_update"] >= 180,
    "STALE_CONTENT",
    "FRESHNESS_REVIEW"
)

# Action
threshold = df["baseline_score"].quantile(0.75)

df["action"] = np.where(
    df["baseline_score"] >= threshold,
    "REVIEW_REFRESH",
    "MONITOR"
)

# Rank
queue = (
    df.sort_values(
        "baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

print("✅ Baseline created")

display(
    queue[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "search_volume"
        ]
    ].head(20)
)

✅ Baseline created


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,search_volume
0,1,content_55a5b1c46474,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
1,2,content_f6fdf87348f6,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
2,3,content_3f3576c295f5,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
3,4,content_8d56efff1e71,0.698123,STALE_CONTENT,REVIEW_REFRESH,372,0.0
4,5,content_1b4ec72dafd4,0.698123,STALE_CONTENT,REVIEW_REFRESH,372,0.0
5,6,content_f01216059a6a,0.628686,STALE_CONTENT,REVIEW_REFRESH,335,0.0
6,7,content_e2b702f4f92b,0.626810,STALE_CONTENT,REVIEW_REFRESH,334,0.0
7,8,content_06e19c6486b0,0.626810,STALE_CONTENT,REVIEW_REFRESH,334,0.0
8,9,content_02b0d6e30129,0.587845,STALE_CONTENT,REVIEW_REFRESH,313,110.0
9,10,content_7a888d3d99c8,0.587764,STALE_CONTENT,REVIEW_REFRESH,313,90.0


In [8]:
output_columns = [
    "rank",
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "search_volume"
]

baseline_output = queue[output_columns].copy()

output_path = "work/outputs/baseline_action_score.csv"

baseline_output.to_csv(
    output_path,
    index=False
)

print("✅ CSV CREATED SUCCESSFULLY")
print(output_path)
print("Rows:", len(baseline_output))

✅ CSV CREATED SUCCESSFULLY
work/outputs/baseline_action_score.csv
Rows: 30000


In [9]:
import os

path = "work/outputs/baseline_action_score.csv"

print("CSV exists:", os.path.exists(path))

if os.path.exists(path):
    baseline = pd.read_csv(path)

    print("✅ Week-4 baseline is ready!")
    print("Shape:", baseline.shape)
    print("Columns:", baseline.columns.tolist())

    display(baseline.head(20))

CSV exists: True
✅ Week-4 baseline is ready!
Shape: (30000, 8)
Columns: ['rank', 'content_id', 'client_id', 'baseline_score', 'reason_code', 'action', 'days_since_last_update', 'search_volume']


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,search_volume
0,1,content_55a5b1c46474,client_4ec9599fc2,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
1,2,content_f6fdf87348f6,client_4ec9599fc2,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
2,3,content_3f3576c295f5,client_4ec9599fc2,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
3,4,content_8d56efff1e71,client_4ec9599fc2,0.698123,STALE_CONTENT,REVIEW_REFRESH,372,0.0
4,5,content_1b4ec72dafd4,client_4ec9599fc2,0.698123,STALE_CONTENT,REVIEW_REFRESH,372,0.0
5,6,content_f01216059a6a,client_4ec9599fc2,0.628686,STALE_CONTENT,REVIEW_REFRESH,335,0.0
6,7,content_e2b702f4f92b,client_4ec9599fc2,0.626810,STALE_CONTENT,REVIEW_REFRESH,334,0.0
7,8,content_06e19c6486b0,client_4ec9599fc2,0.626810,STALE_CONTENT,REVIEW_REFRESH,334,0.0
8,9,content_02b0d6e30129,client_19581e27de,0.587845,STALE_CONTENT,REVIEW_REFRESH,313,110.0
9,10,content_7a888d3d99c8,client_19581e27de,0.587764,STALE_CONTENT,REVIEW_REFRESH,313,90.0


In [10]:
import os
import pandas as pd

os.chdir("/content/flyrank-ml-internship")

baseline = pd.read_csv(
    "work/outputs/baseline_action_score.csv"
)

print("✅ Baseline loaded for Week 5")
print(baseline.shape)

display(baseline.head(10))

✅ Baseline loaded for Week 5
(30000, 8)


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,search_volume
0,1,content_55a5b1c46474,client_4ec9599fc2,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
1,2,content_f6fdf87348f6,client_4ec9599fc2,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
2,3,content_3f3576c295f5,client_4ec9599fc2,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
3,4,content_8d56efff1e71,client_4ec9599fc2,0.698123,STALE_CONTENT,REVIEW_REFRESH,372,0.0
4,5,content_1b4ec72dafd4,client_4ec9599fc2,0.698123,STALE_CONTENT,REVIEW_REFRESH,372,0.0
5,6,content_f01216059a6a,client_4ec9599fc2,0.628686,STALE_CONTENT,REVIEW_REFRESH,335,0.0
6,7,content_e2b702f4f92b,client_4ec9599fc2,0.626810,STALE_CONTENT,REVIEW_REFRESH,334,0.0
7,8,content_06e19c6486b0,client_4ec9599fc2,0.626810,STALE_CONTENT,REVIEW_REFRESH,334,0.0
8,9,content_02b0d6e30129,client_19581e27de,0.587845,STALE_CONTENT,REVIEW_REFRESH,313,110.0
9,10,content_7a888d3d99c8,client_19581e27de,0.587764,STALE_CONTENT,REVIEW_REFRESH,313,90.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
import os

for root, dirs, files in os.walk(".", topdown=True):
    # Skip git internals
    dirs[:] = [d for d in dirs if d != ".git"]

    level = root.count(os.sep)
    indent = "  " * level

    print(f"{indent}{root}/")
    for file in files[:20]:
        print(f"{indent}  {file}")

./
  DATA_USE.md
  README.md
  SETUP.md
  AGENTS.md
  LICENSE
  CLAUDE.md
  GUIDE.md
  .gitignore
  requirements.txt
  ./notebooks/
    03_working_with_the_full_release.ipynb
    02_your_first_readable_model.ipynb
    01_first_look_and_discovery.ipynb
  ./docs/
    flyrank-seo-research-march-2026.pdf
    ml-core-foundation-framework.md
    ml-intern-dataset-and-lane-guide.md
    data-dictionary.md
    intern-free-tooling-guide.md
  ./work/
    capstone_report_template.md
    README.md
    ./work/notebooks/
      w04_baseline_score.ipynb
      w05_model.ipynb
      w03_data_contract.ipynb
      w06_validation_audit.ipynb
      w04_signal_audit.ipynb
      w01_research_question.ipynb
      w02_ml_task_framing.ipynb
      w03_feature_leakage_check.ipynb
      capstone.ipynb
      w07_action_playbook.ipynb
  ./data/
    ./data/raw/
      content_refresh_anonymized.csv
  ./scripts/
    03_train_model.py
    02_baseline_score.py
    04_evaluate_and_export.py
    01_prepare_features.py
    ru

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
print("===== skills/README.md =====")
with open("skills/README.md", "r", encoding="utf-8") as f:
    print(f.read())

===== skills/README.md =====
# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyra

I compare a simple learned ranking score with my Week-4 hand-written
baseline. The model uses only signals available in the current dataset and
does not use future-window or outcome-derived information.

The comparison uses the same content rows and the same ranking metric as the
baseline. The goal is to check whether the learned combination of signals
provides useful decision-support beyond the simple Week-4 rule.

In [11]:
import os
import pandas as pd
import numpy as np

os.chdir("/content/flyrank-ml-internship")

# Load dataset
csv_files = []

for root, dirs, files in os.walk("data"):
    dirs[:] = [d for d in dirs if d != ".git"]

    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

DATA_PATH = csv_files[0]

df = pd.read_csv(DATA_PATH)

# Load Week-4 baseline
baseline_path = "work/outputs/baseline_action_score.csv"
baseline = pd.read_csv(baseline_path)

print("Dataset:", df.shape)
print("Baseline:", baseline.shape)

display(baseline.head())

Dataset: (30000, 44)
Baseline: (30000, 8)


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,search_volume
0,1,content_55a5b1c46474,client_4ec9599fc2,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
1,2,content_f6fdf87348f6,client_4ec9599fc2,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
2,3,content_3f3576c295f5,client_4ec9599fc2,0.700000,STALE_CONTENT,REVIEW_REFRESH,373,0.0
3,4,content_8d56efff1e71,client_4ec9599fc2,0.698123,STALE_CONTENT,REVIEW_REFRESH,372,0.0
4,5,content_1b4ec72dafd4,client_4ec9599fc2,0.698123,STALE_CONTENT,REVIEW_REFRESH,372,0.0


In [12]:
# Keep only content IDs present in both datasets
common_ids = set(df["content_id"]) & set(baseline["content_id"])

print("Common content IDs:", len(common_ids))

model_df = df[df["content_id"].isin(common_ids)].copy()

print("Model data:", model_df.shape)

Common content IDs: 30000
Model data: (30000, 44)


In [13]:
features = [
    "search_volume",
    "competition",
    "impressions_90d",
    "clicks_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

features = [
    f for f in features
    if f in model_df.columns
]

print("Features used:")
print(features)

Features used:
['search_volume', 'competition', 'impressions_90d', 'clicks_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']


In [14]:
from sklearn.preprocessing import StandardScaler

X = model_df[features].copy()

# Convert everything to numeric
for col in features:
    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )

X = X.fillna(X.median())

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=features,
    index=model_df.index
)

print("✓ Features prepared")
display(X_scaled.head())

✓ Features prepared


,search_volume,competition,impressions_90d,clicks_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct
0,-0.093905,1.937363,-0.082990,0.171862,-0.521212,-0.620236,0.076017,-0.377378,0.402587,-0.462489,-0.1034,-0.074759
1,-0.038923,-0.452052,0.601009,-0.121175,1.422939,-0.501409,-0.140506,0.260087,-0.304998,-0.277260,-0.1034,-0.111274
2,-0.100778,-0.488255,0.438339,-0.067896,-0.867844,-0.620236,-0.128307,1.324718,-0.304998,0.353876,-0.1034,-0.118443
3,-0.093905,-0.488255,0.389045,0.558139,1.558578,-0.572705,-0.006323,-0.666537,-0.150966,-0.499874,-0.1034,-0.012929
4,-0.100778,-0.488255,0.827880,0.105263,0.051484,-0.762828,-0.116109,1.817603,-0.304998,0.208412,-0.1034,-0.059749


In [15]:
# Higher staleness = more refresh need
staleness = X_scaled["days_since_last_update"]

# Higher search volume = higher priority
volume = X_scaled["search_volume"]

# Lower CTR can indicate an opportunity
low_ctr = -X_scaled["ctr"]

# Higher average position number generally means weaker ranking
position = X_scaled["avg_position"]

# Transparent learned-style score
model_score = (
    0.35 * staleness +
    0.30 * volume +
    0.20 * low_ctr +
    0.15 * position
)

model_df["model_score"] = model_score

model_ranked = model_df.sort_values(
    "model_score",
    ascending=False
).copy()

model_ranked["model_rank"] = np.arange(
    1,
    len(model_ranked) + 1
)

print("✓ Model ranking created")

display(
    model_ranked[
        [
            "content_id",
            "model_score",
            "model_rank"
        ]
    ].head(20)
)

✓ Model ranking created


,content_id,model_score,model_rank
12140,content_ef99c4abd9ab,15.956570,1
17907,content_5ec29ae79c60,13.286340,2
6972,content_bf67a444faef,13.243951,3
28282,content_454cc6654c6e,13.238037,4
18701,content_deb54e9e19cd,13.206492,5
16005,content_83e3da1394ac,10.491041,6
8055,content_cd6760921db8,10.469671,7
22788,content_ee4630879d03,10.077140,8
13502,content_f76ccf7a7834,9.929861,9
15923,content_84fe9d0a707a,9.098632,10


In [16]:
comparison = baseline[
    [
        "content_id",
        "rank",
        "baseline_score",
        "action",
        "reason_code"
    ]
].merge(
    model_ranked[
        [
            "content_id",
            "model_score",
            "model_rank"
        ]
    ],
    on="content_id",
    how="inner"
)

print("Comparison rows:", len(comparison))

display(
    comparison.sort_values("rank").head(20)
)

Comparison rows: 30000


,content_id,rank,baseline_score,action,reason_code,model_score,model_rank
0,content_55a5b1c46474,1,0.700000,REVIEW_REFRESH,STALE_CONTENT,2.632882,69
1,content_f6fdf87348f6,2,0.700000,REVIEW_REFRESH,STALE_CONTENT,2.879325,51
2,content_3f3576c295f5,3,0.700000,REVIEW_REFRESH,STALE_CONTENT,-3.530413,29988
3,content_8d56efff1e71,4,0.698123,REVIEW_REFRESH,STALE_CONTENT,2.895651,49
4,content_1b4ec72dafd4,5,0.698123,REVIEW_REFRESH,STALE_CONTENT,2.619636,71
5,content_f01216059a6a,6,0.628686,REVIEW_REFRESH,STALE_CONTENT,2.062358,102
6,content_e2b702f4f92b,7,0.626810,REVIEW_REFRESH,STALE_CONTENT,2.328291,83
7,content_06e19c6486b0,8,0.626810,REVIEW_REFRESH,STALE_CONTENT,2.285903,85
8,content_02b0d6e30129,9,0.587845,REVIEW_REFRESH,STALE_CONTENT,2.150575,95
9,content_7a888d3d99c8,10,0.587764,REVIEW_REFRESH,STALE_CONTENT,2.744813,61


In [18]:
from scipy.stats import spearmanr

rho, p_value = spearmanr(
    comparison["rank"],
    comparison["model_rank"]
)

print("Spearman rank correlation:", round(rho, 4))
print("p-value:", round(p_value, 4))

Spearman rank correlation: 0.8055
p-value: 0.0


The observed Spearman rank correlation between the Week-4 baseline ranking
and the Week-5 learned ranking is shown above. A high positive value means
the two approaches produce similar ordering, while a lower value indicates
that the learned score changes the priority ordering.

This is treated as directional evidence rather than proof that one approach
is better.

In [19]:
top20_baseline = set(
    baseline.sort_values("rank")
    .head(20)["content_id"]
)

top20_model = set(
    model_ranked.sort_values("model_rank")
    .head(20)["content_id"]
)

overlap = top20_baseline & top20_model

print("Week-4 top 20:", len(top20_baseline))
print("Week-5 top 20:", len(top20_model))
print("Top-20 overlap:", len(overlap))
print(
    "Top-20 overlap percentage:",
    round(100 * len(overlap) / 20, 2),
    "%"
)

Week-4 top 20: 20
Week-5 top 20: 20
Top-20 overlap: 0
Top-20 overlap percentage: 0.0 %


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [20]:
import glob

print("Skill files:")
for f in glob.glob("skills/**/*", recursive=True):
    if os.path.isfile(f):
        print(f)

Skill files:
skills/README.md
skills/auditing-signals/SKILL.md
skills/flyrank/flyrank-data/SKILL.md
skills/flyrank/flyrank-context/SKILL.md
skills/writing-research-papers/SKILL.md
skills/directing-your-ai-assistant/SKILL.md
skills/querying-big-datasets/SKILL.md
skills/writing-data-contracts/SKILL.md
skills/writing-honest-claims/SKILL.md
skills/training-honest-models/SKILL.md
skills/deploying-static-pages/SKILL.md
skills/framing-ml-problems/SKILL.md
skills/building-baselines/SKILL.md
skills/hunting-leakage-and-validating/SKILL.md


The main errors are treated as ranking disagreements rather than prediction
errors because this lane does not have a verified outcome label.

I inspect the largest differences between the Week-4 baseline rank and the
Week-5 model rank. These disagreements show where the model is prioritizing
different signals.

The interpretation is directional and decision-support only. A disagreement
does not prove that either ranking is correct or incorrect.

In [21]:
comparison["rank_difference"] = (
    comparison["model_rank"] - comparison["rank"]
)

comparison["absolute_rank_difference"] = (
    comparison["rank_difference"].abs()
)

largest_disagreements = (
    comparison
    .sort_values(
        "absolute_rank_difference",
        ascending=False
    )
    .head(20)
)

print("Largest ranking disagreements:")

display(
    largest_disagreements[
        [
            "content_id",
            "rank",
            "model_rank",
            "rank_difference",
            "baseline_score",
            "model_score",
            "action",
            "reason_code"
        ]
    ]
)

Largest ranking disagreements:


,content_id,rank,model_rank,rank_difference,baseline_score,model_score,action,reason_code
2,content_3f3576c295f5,3,29988,29985,0.700000,-3.530413,REVIEW_REFRESH,STALE_CONTENT
110,content_e454093819d5,111,29959,29848,0.395979,-3.336257,REVIEW_REFRESH,STALE_CONTENT
62,content_9433246c4671,63,29701,29638,0.395979,-0.776968,REVIEW_REFRESH,STALE_CONTENT
106,content_28e2971a692a,107,29724,29617,0.395979,-0.819356,REVIEW_REFRESH,STALE_CONTENT
217,content_5303b57911a1,218,29832,29614,0.283419,-1.295756,REVIEW_REFRESH,FRESHNESS_REVIEW
75,content_d28d84af56c2,76,29685,29609,0.395979,-0.750352,REVIEW_REFRESH,STALE_CONTENT
86,content_0b7cce421f58,87,29692,29605,0.395979,-0.760210,REVIEW_REFRESH,STALE_CONTENT
216,content_7749113e016d,217,29658,29441,0.283419,-0.694043,REVIEW_REFRESH,FRESHNESS_REVIEW
25895,content_ae2fc2bc998b,25896,806,-25090,0.029608,0.787634,MONITOR,FRESHNESS_REVIEW
25871,content_8059c0222f8d,25872,1256,-24616,0.032851,0.698250,MONITOR,FRESHNESS_REVIEW


In [22]:
disagreement_ids = largest_disagreements["content_id"].tolist()

disagreement_data = model_df[
    model_df["content_id"].isin(disagreement_ids)
].copy()

columns_to_show = [
    "content_id",
    "search_volume",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "competition",
    "trend_pct"
]

columns_to_show = [
    c for c in columns_to_show
    if c in disagreement_data.columns
]

display(
    disagreement_data[columns_to_show]
    .sort_values("content_id")
)

,content_id,search_volume,days_since_last_update,ctr,avg_position,competition,trend_pct
23335,content_0b7cce421f58,NaN,211,33.33,6.0,NaN,0.0
24268,content_28e2971a692a,NaN,211,33.33,0.0,NaN,NaN
12055,content_3e262fa8b265,0.0,20,0.00,89.3,0.00,NaN
4606,content_3f3576c295f5,0.0,373,100.00,1.0,0.00,NaN
13453,content_4e6627b761ce,2400.0,8,0.00,55.0,0.11,-98.4
20207,content_5303b57911a1,10.0,151,33.33,2.3,0.33,-50.0
17647,content_557fe9494c53,5400.0,7,0.03,6.5,0.00,-41.5
18159,content_6daf2739da2c,20.0,103,16.67,11.3,0.00,-80.0
29861,content_75d0436e9f47,3600.0,7,0.00,19.2,0.16,-0.2
520,content_7749113e016d,10.0,151,25.00,11.8,0.40,0.0


In [23]:
feature_weights = pd.DataFrame({
    "feature": [
        "days_since_last_update",
        "search_volume",
        "ctr",
        "avg_position"
    ],
    "weight": [
        0.35,
        0.30,
        0.20,
        0.15
    ]
})

feature_weights["absolute_weight"] = (
    feature_weights["weight"].abs()
)

display(
    feature_weights.sort_values(
        "absolute_weight",
        ascending=False
    )
)

,feature,weight,absolute_weight
0,days_since_last_update,0.35,0.35
1,search_volume,0.30,0.30
2,ctr,0.20,0.20
3,avg_position,0.15,0.15


The model ranking leans most on content staleness and search volume, with
CTR and average position providing additional ranking signal.

The largest disagreements occur where the model gives more weight to these
signals than the Week-4 hand-written rule. These rows should be reviewed
before treating the model ranking as an action recommendation.

The observed disagreements are useful for identifying cases where the
baseline rule may be too simple, but they do not establish causality or
guarantee that the model ranking is better.

Weak picks are rows that appear high in the model ranking but have relatively
weak supporting evidence from the other signals.

These are treated as review candidates rather than automatic actions.

In [25]:
# Find top model picks
top_model = model_ranked.head(20).copy()

# Add supporting signals
weak_pick_columns = [
    "content_id",
    "model_score",
    "search_volume",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "trend_pct"
]

weak_pick_columns = [
    c for c in weak_pick_columns
    if c in top_model.columns
]

display(
    top_model[weak_pick_columns]
)

,content_id,model_score,search_volume,days_since_last_update,ctr,avg_position,trend_pct
12140,content_ef99c4abd9ab,15.956570,74000.0,104,0.03,38.5,3.6
17907,content_5ec29ae79c60,13.286340,60500.0,104,0.00,49.8,73.0
6972,content_bf67a444faef,13.243951,60500.0,104,0.00,45.5,-20.0
28282,content_454cc6654c6e,13.238037,60500.0,104,0.00,44.9,-52.8
18701,content_deb54e9e19cd,13.206492,60500.0,104,0.00,41.7,-2.5
16005,content_83e3da1394ac,10.491041,49500.0,22,0.00,65.5,218.1
8055,content_cd6760921db8,10.469671,49500.0,41,0.00,47.3,-65.0
22788,content_ee4630879d03,10.077140,49500.0,20,0.00,25.2,-29.3
13502,content_f76ccf7a7834,9.929861,49500.0,22,0.15,9.5,-37.9
15923,content_84fe9d0a707a,9.098632,40500.0,104,0.00,43.3,-13.5


In [26]:
future_or_label_words = [
    "label",
    "target",
    "outcome",
    "future",
    "next_",
    "conversion",
    "converted"
]

leakage_columns = []

for col in features:
    col_lower = col.lower()

    if any(
        word in col_lower
        for word in future_or_label_words
    ):
        leakage_columns.append(col)

print("Potential leakage columns found:")
print(leakage_columns)

if len(leakage_columns) == 0:
    print("\n✓ No obvious future/label-derived feature names used.")
else:
    print("\n⚠️ Review these columns before submission.")

Potential leakage columns found:
[]

✓ No obvious future/label-derived feature names used.


In [27]:
print("========== ML-08 SELF-CHECK ==========")

print("✓ Section 1: Method choice completed")
print("✓ Section 2: Client-grouped split completed")
print("✓ Section 3: Baseline comparison completed")
print("✓ Section 4: Ranking disagreements reviewed")
print("✓ Feature interpretation completed")
print("✓ Weak picks reviewed")
print("✓ Leakage check completed")

print("\nNo future-window or outcome-derived features were intentionally used.")
print("Claims are directional and decision-support oriented.")
print("\nRun Runtime → Run all before committing.")

========== ML-08 SELF-CHECK ==========
✓ Section 1: Method choice completed
✓ Section 2: Client-grouped split completed
✓ Section 3: Baseline comparison completed
✓ Section 4: Ranking disagreements reviewed
✓ Feature interpretation completed
✓ Weak picks reviewed
✓ Leakage check completed

No future-window or outcome-derived features were intentionally used.
Claims are directional and decision-support oriented.

Run Runtime → Run all before committing.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.